# 03 — MVTec AD 2: confirmation, diagnosis, localization rebuilt, and the v6 submission

The whole AD 2 line in order: fetch (§1), the confirmation of E13 (§2), the components of v5
on `test_public` (§3), the fusion and mask-rule studies of E21 (§4), the learned head of E23
(§5), and the export of v5 and v6 for MVTec's server (§6). Reports: `reports/11`, `12`, `13`
and `00-final-report.md`.

Every heavy step is switched off in `RUN` and only prints its command until switched on. The
whole line fits the 4 GB laptop: about 2 s per image for the DINOv3 component, ~10 h for the
v5 export, ~6 h for the E23 features. Reruns write under `artifacts/ad2/` (not tracked), never
over the committed study tables.

Data: MVTec AD 2 © MVTec Software GmbH, CC BY-NC-SA 4.0, evaluation only; no image, map or
mask leaves `data/` or `artifacts/`. The project's other datasets: VisA © Amazon, CC BY 4.0;
MPDD, CC BY-NC-SA 4.0. See [ATTRIBUTION.md](../ATTRIBUTION.md).

## 0. Environment

The DINOv3 ViT-H+ component runs in bf16 (fp16 overflows) and needs about 2.7 GB of VRAM.

In [ ]:
import os, sys, json, pathlib, subprocess

REPO = pathlib.Path.cwd()
if not (REPO / 'src' / 'inspector').exists():
    REPO = REPO.parent  # running from notebooks/
os.chdir(REPO)  # scripts and configs are addressed from the repository root
sys.path.insert(0, str(REPO / 'src'))

from inspector.utils.env import capture

env = capture()
print('git      :', env['git_sha'], '(dirty)' if env['dirty'] else '(clean)')
print('device   :', env['device'].get('gpu_name', 'CPU only'),
      f"{env['device'].get('gpu_total_mb', 0)} MB")

def sh(*args, run=True):
    """Run a repository script (or `-m inspector ...`) with this interpreter.
    The command is always printed; with run=False it is only printed."""
    cmd = [sys.executable, *map(str, args)]
    print('$ python', ' '.join(cmd[1:]), '' if run else '   (not run)')
    if run:
        subprocess.run(cmd, check=True)

In [ ]:
AD2 = pathlib.Path(os.environ.get('MVI_AD2_ROOT', 'data/raw/mvtec_ad2'))
WORK = pathlib.Path(os.environ.get('MVI_AD2_WORK', 'artifacts/ad2'))  # outside git
CATEGORIES = ['can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts']
RUN = {'fetch': False, 'e13': False, 'components': False, 'e21': False, 'e23': False, 'export': False}
WORK.mkdir(parents=True, exist_ok=True)

from inspector.runner import implementation_id

# runs of this implementation by default; set MVI_IMPLEMENTATION to analyse earlier runs
IMPL = os.environ.get('MVI_IMPLEMENTATION', implementation_id())

def run_ids(match):
    """category -> id of the completed AD 2 run of implementation IMPL that `match` accepts."""
    out = {}
    for spec_path in sorted((REPO / 'reports' / 'runs').glob('*/spec.json')):
        spec = json.loads(spec_path.read_text(encoding='utf-8'))
        result = spec_path.with_name('result.json')
        if (str(spec.get('implementation_id', '')).startswith(IMPL[:12])
                and spec['dataset'].get('layout') == 'mvtec_ad2' and result.exists()
                and json.loads(result.read_text(encoding='utf-8')).get('status') == 'completed'
                and match(spec)):
            out[spec['dataset']['category']] = spec_path.parent.name
    return out

def is_dino(spec):
    return spec['model']['name'] == 'tiled_knn' and 'huge_plus' in spec['model']['hparams']['backbone']

def is_wrn1280(spec):
    return spec['model']['name'] == 'patchcore' and spec['transform']['long_side'] == 1280

def ids(found):
    return [found[c] for c in CATEGORIES if c in found]

## 1. Data

Eight categories, fetched one at a time and verified against a SHA-256 manifest (`-I`: the
interpreter does not import from the working directory while it handles downloaded files).

In [ ]:
sh('-I', 'scripts/data_fetch_mvtec_ad2.py', '--out', AD2, run=RUN['fetch'])

## 2. Confirmation v2 (E13, ADR-13)

The VisA-selected candidate v2 (`layer2`, 960 px) against recipe v1 on `test_public`, three
seeds, then the analysis that was pre-registered before the download. Outcome: v2 not
confirmed, and both recipes collapse under the changed lighting (`reports/11`).

In [ ]:
for recipe in ('confirmation-v1-patchcore-640', 'candidate-v2-patchcore-layer2-960'):
    sh('-m', 'inspector', 'run', '-c', 'configs/data/mvtec_ad2.yaml', '-c', f'configs/recipes/{recipe}.yaml',
       '--data-root', AD2, '--role', 'confirmation', '--categories', *CATEGORIES, '--seeds', 0, 1, 2,
       run=RUN['e13'])
sh('scripts/ad2_e13_confirmation_analysis.py', '--out', WORK / 'ad2-confirmation', run=RUN['e13'])

## 3. The two components of v5 on `test_public`

Tiled DINOv3 ViT-H+/16 kNN (ADR-16) and PatchCore WRN50 at 1280 px (recipe v3), seed 0,
every category. Their stored maps feed every study below; the server proxy scores them.

In [ ]:
for recipe in ('candidate-v5-component-dinov3hplus', 'candidate-v3-patchcore-1280'):
    sh('-m', 'inspector', 'run', '-c', 'configs/data/mvtec_ad2.yaml', '-c', f'configs/recipes/{recipe}.yaml',
       '--data-root', AD2, '--categories', *CATEGORIES, '--seeds', 0, run=RUN['components'])
DINO, WRN = run_ids(is_dino), run_ids(is_wrn1280)
print('DINOv3 ViT-H+ runs:', DINO)
print('WRN50 1280 runs   :', WRN)

## 4. E21: fusion and the mask rule

Each map divided by its validation P95, maximum of the two, then the mask rules over threshold
gains 1.1-1.6 (with the oriented line closing and the image mark), and the split-half test of
one global rule against per-category rules.

In [ ]:
run = RUN['e21'] and len(DINO) == len(WRN) == len(CATEGORIES)
sh('scripts/ad2_e21_fusion.py', '--data-root', AD2, '--a', *ids(DINO), '--b', *ids(WRN), '--mode', 'max',
   '--gains', '--label', 'hplus-max-wrn1280', '--out', WORK / 'fusion',
   '--dump', WORK / 'fusion' / 'dump-hplus-max-wrn1280', run=run)
sh('scripts/ad2_e21_threshold_transfer.py', '--dump', WORK / 'fusion' / 'dump-hplus-max-wrn1280',
   '--out', WORK / 'fusion' / 'transfer.json', run=run)
sh('scripts/server_proxy.py', '--data-root', AD2, '--runs', *ids(WRN), '--label', 'wrn1280',
   '--out', WORK / 'ad2-server', run=run)

## 5. E23: the learned head for the maps

Both components refitted on seven eighths of `train/good`; the held-out eighth, each image
also with three synthetic defects, trains a per-category logistic head. Then the head applied
to the full-bank maps exactly as the export builds them, and the object bootstrap of its gain.

In [ ]:
FEATURES = WORK / 'head_features'
run = RUN['e23']
sh('scripts/ad2_e23_head_features.py', '--data-root', AD2, '--out', FEATURES, run=run)
sh('scripts/ad2_e23_fusion_head.py', '--features', FEATURES, '--out', WORK / 'fusion-head',
   '--apply-dino', *ids(DINO), '--apply-wrn', *ids(WRN), run=run)
for script, name in (('ad2_e23_hybrid_check.py', 'hybrid_check.json'),
                     ('ad2_e23_hybrid_bootstrap.py', 'hybrid_bootstrap.json')):
    sh(f'scripts/{script}', '--features', FEATURES, '--dino', *ids(DINO), '--wrn', *ids(WRN),
       '--out', WORK / 'fusion-head' / name, run=run)

## 6. Export: v5, then v6 (head maps with v5's masks)

The v5 export fits every category on its full training split and writes native-resolution
float16 maps and PNG masks for `test_private` and `test_private_mixed`, keeping each
component's map. v6 replaces the maps with the head's on the CPU and copies v5's masks byte
for byte. MVTec's `check_and_prepare_data_for_upload.py` (from the benchmark's public code)
validates and compresses a submission directory; it accepts only the two map folders, so the
kept components are moved out first.

In [ ]:
run = RUN['export']
sh('scripts/server_export_submission.py', '--data-root', AD2,
   '--recipe', 'configs/recipes/candidate-v5-fusion.yaml', '--out', WORK / 'v5', '--keep-components', run=run)
if run and (WORK / 'v5' / '_components').exists():
    (WORK / 'v5' / '_components').rename(WORK / 'components')
sh('scripts/server_export_v6_hybrid.py', '--features', FEATURES, '--components', WORK / 'components',
   '--masks', WORK / 'v5', '--out', WORK / 'v6', run=run)